# Library load

In [1]:
import json
from datetime import datetime, timezone
import pandas as pd

# Read needed file

In [7]:
path = "logs-mhs-wenyi222.json"

with open(path, "r", encoding="utf-8") as f:
    records = json.load(f)

type(records), len(records), records[0].keys()

(list,
 2360,
 dict_keys(['_id', 'data', 'device', 'eventType', 'game', 'playerId', 'sceneName', 'serverTimestamp', 'version']))

## Unique event types

In [8]:
unique_event_types = sorted({
    r.get("eventType")
    for r in records
    if r.get("eventType") is not None
})

unique_event_types

['DialogueEvent',
 'EndOfUnit',
 'PlayerPositionEvent',
 'Soil Key Puzzle',
 'TerasGardenBox',
 'Topographic Map Event',
 'TopographicMapEvent',
 'WaterChamberEvent',
 'argumentationEvent',
 'argumentationNodeEvent',
 'argumentationToolEvent',
 'questEvent',
 'soilMachine']

## Unique scene names

In [10]:
unique_scene_types = sorted({
    r.get("sceneName")
    for r in records
    if r.get("sceneName") is not None
})

unique_scene_types

['Unit 1 Dev',
 'Unit 2 Prod (Refactor)',
 'Unit 3 Dev',
 'Unit 3 Dungeon Dev',
 'Unit 4 Dev',
 'Unit 4 Dev - Anderson Base',
 'Unit 4 Dev - Dungeon',
 'Unit 5 Dev',
 'Unit 5 Dev - Dungeon']

# Checking problem PPs

In [11]:
pid = "wenyi222"

## U4 P1
### No time window

In [13]:
CORRECT_KEY = "DialogueNodeEvent:88:5"

EVENT_TYPE = "Soil Key Puzzle"
START_STATUS = "Started"
END_STATUS = "Finished"

score = 0.0

# Check whether correct dialogue exists
has_8805 = any(
    rec.get("game") == "mhs" and
    rec.get("playerId") == pid and
    rec.get("eventKey") == CORRECT_KEY
    for rec in records
)

if has_8805:
    score += 0.5

# Find earliest start_doc
start_candidates = [
    rec for rec in records
    if rec.get("game") == "mhs"
    and rec.get("playerId") == pid
    and rec.get("eventType") == EVENT_TYPE
    and rec.get("data", {}).get("Soil Key Puzzle Status") == START_STATUS
    and "serverTimestamp" in rec
]

start_doc = min(start_candidates, key=lambda x: x["serverTimestamp"], default=None)

duration_seconds = None

if start_doc is not None:
    # Find earliest end_doc
    end_candidates = [
        rec for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventType") == EVENT_TYPE
        and rec.get("data", {}).get("Soil Key Puzzle Status") == END_STATUS
        and "serverTimestamp" in rec
    ]

    end_doc = min(end_candidates, key=lambda x: x["serverTimestamp"], default=None)

    if end_doc is not None:
        start_ts = datetime.fromisoformat(start_doc["serverTimestamp"].replace("Z", "+00:00"))
        end_ts = datetime.fromisoformat(end_doc["serverTimestamp"].replace("Z", "+00:00"))
        duration_seconds = (end_ts - start_ts).total_seconds()

if duration_seconds is not None:
    if 0 < duration_seconds <= 30:
        score += 1.0
    elif 30 < duration_seconds <= 90:
        score += 0.5

color = "green" if score >= 1 else "yellow"

color

'green'

### Production codes

In [16]:
TRIGGER_KEY = "questActiveEvent:39"
CORRECT_KEY = "DialogueNodeEvent:88:5"

EVENT_TYPE = "Soil Key Puzzle"
START_STATUS = "Started"
END_STATUS = "Finished"

# 1) Latest trigger (end anchor)
trigger_docs = sorted(
    [
        rec for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") == TRIGGER_KEY
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_trigger = trigger_docs[-1] if trigger_docs else None

if latest_trigger is None:
    color = "yellow"
else:
    # 2) Previous trigger (attempt boundary)
    prev_trigger = trigger_docs[-2] if len(trigger_docs) >= 2 else None

    window_start_id = prev_trigger["_id"] if prev_trigger is not None else "000000000000000000000000"
    window_end_id = latest_trigger["_id"]

    score = 0.0

    # +0.5 if correct choice selected inside window
    has_8805 = any(
        rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") == CORRECT_KEY
        and window_start_id < rec.get("_id", "") <= window_end_id
        for rec in records
    )

    if has_8805:
        score += 0.5

    # Find Started inside window (earliest)
    start_candidates = sorted(
        [
            rec for rec in records
            if rec.get("game") == "mhs"
            and rec.get("playerId") == pid
            and rec.get("eventType") == EVENT_TYPE
            and rec.get("data", {}).get("Soil Key Puzzle Status") == START_STATUS
            and window_start_id < rec.get("_id", "") <= window_end_id
            and "serverTimestamp" in rec
            and "_id" in rec
        ],
        key=lambda x: x["_id"]
    )

    start_doc = start_candidates[0] if start_candidates else None
    duration_seconds = None

    if start_doc is not None and "serverTimestamp" in start_doc:
        # Find Finished AFTER startDoc, still within window
        end_candidates = sorted(
            [
                rec for rec in records
                if rec.get("game") == "mhs"
                and rec.get("playerId") == pid
                and rec.get("eventType") == EVENT_TYPE
                and rec.get("data", {}).get("Soil Key Puzzle Status") == END_STATUS
                and start_doc["_id"] < rec.get("_id", "") <= window_end_id
                and "serverTimestamp" in rec
            ],
            key=lambda x: x["_id"]
        )

        end_doc = end_candidates[0] if end_candidates else None

        if end_doc is not None and "serverTimestamp" in end_doc:
            start_ts = datetime.fromisoformat(start_doc["serverTimestamp"].replace("Z", "+00:00"))
            end_ts = datetime.fromisoformat(end_doc["serverTimestamp"].replace("Z", "+00:00"))
            duration_seconds = (end_ts - start_ts).total_seconds()

    # Duration scoring
    if duration_seconds is not None:
        if 0 < duration_seconds <= 30:
            score += 1.0
        elif 30 < duration_seconds <= 90:
            score += 0.5

    color = "green" if score >= 1 else "yellow"

score, color

(1.0, 'green')

## U4 P4
### No time window

In [51]:
score = 0

# Count machine 1 / floor 5 / TopRow
c_m1_top = sum(
    1 for rec in records
    if rec.get("playerId") == pid
    and rec.get("eventType") == "soilMachine"
    and rec.get("data", {}).get("floor") == "5"
    and rec.get("data", {}).get("machine") == "1"
    and rec.get("data", {}).get("row") == "TopRow"
)

# Count machine 1 / floor 5 / BottomRow
c_m1_bottom = sum(
    1 for rec in records
    if rec.get("playerId") == pid
    and rec.get("eventType") == "soilMachine"
    and rec.get("data", {}).get("floor") == "5"
    and rec.get("data", {}).get("machine") == "1"
    and rec.get("data", {}).get("row") == "BottomRow"
)

if c_m1_top == 1 and c_m1_bottom == 1:
    score += 1

# Count machine 2 / floor 5
c_m2_floor5 = sum(
    1 for rec in records
    if rec.get("playerId") == pid
    and rec.get("eventType") == "soilMachine"
    and rec.get("data", {}).get("floor") == "5"
    and rec.get("data", {}).get("machine") == "2"
)

if c_m2_floor5 == 1:
    score += 1

SUCCESS_KEYS = ["DialogueNodeEvent:107:4", "DialogueNodeEvent:107:5"]
NEG_KEYS = ["DialogueNodeEvent:107:2", "DialogueNodeEvent:107:3", "DialogueNodeEvent:107:6"]

success_total = sum(
    1 for rec in records
    if rec.get("playerId") == pid
    and rec.get("eventKey") in SUCCESS_KEYS
)

neg_total = sum(
    1 for rec in records
    if rec.get("playerId") == pid
    and rec.get("eventKey") in NEG_KEYS
)

if success_total == 1 and neg_total == 0:
    score += 2
elif success_total == 1 and neg_total == 1:
    score += 1

color = "green" if score > 2 else "yellow"

print("c_m1_top:", c_m1_top)
print("c_m1_bottom:", c_m1_bottom)
print("c_m2_floor5:", c_m2_floor5)
print("success_total:", success_total)
print("neg_total:", neg_total)
print("score:", score)
print("color:", color)

c_m1_top: 1
c_m1_bottom: 1
c_m2_floor5: 1
success_total: 1
neg_total: 0
score: 4
color: green


### With time window

In [52]:
WINDOW_START_KEY = "questActiveEvent:50"
WINDOW_END_KEY = "questActiveEvent:36"

# 1) Find latest window start
start_candidates = sorted(
    [
        rec for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") == WINDOW_START_KEY
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_start = start_candidates[-1] if start_candidates else None

# 2) Find latest window end
end_candidates = sorted(
    [
        rec for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") == WINDOW_END_KEY
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_end = end_candidates[-1] if end_candidates else None

if latest_start is None or latest_end is None or latest_end["_id"] <= latest_start["_id"]:
    color = "yellow"
else:
    window_start_id = latest_start["_id"]
    window_end_id = latest_end["_id"]

    score = 0

    # Count machine 1 / floor 5 / TopRow inside window
    c_m1_top = sum(
        1 for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventType") == "soilMachine"
        and rec.get("data", {}).get("floor") == "5"
        and rec.get("data", {}).get("machine") == "1"
        and rec.get("data", {}).get("row") == "TopRow"
        and window_start_id < rec.get("_id", "") <= window_end_id
    )

    # Count machine 1 / floor 5 / BottomRow inside window
    c_m1_bottom = sum(
        1 for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventType") == "soilMachine"
        and rec.get("data", {}).get("floor") == "5"
        and rec.get("data", {}).get("machine") == "1"
        and rec.get("data", {}).get("row") == "BottomRow"
        and window_start_id < rec.get("_id", "") <= window_end_id
    )

    if c_m1_top == 1 and c_m1_bottom == 1:
        score += 1

    # Count machine 2 / floor 5 inside window
    c_m2_floor5 = sum(
        1 for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventType") == "soilMachine"
        and rec.get("data", {}).get("floor") == "5"
        and rec.get("data", {}).get("machine") == "2"
        and window_start_id < rec.get("_id", "") <= window_end_id
    )

    if c_m2_floor5 == 1:
        score += 1

    SUCCESS_KEYS = ["DialogueNodeEvent:107:4", "DialogueNodeEvent:107:5"]
    NEG_KEYS = ["DialogueNodeEvent:107:2", "DialogueNodeEvent:107:3", "DialogueNodeEvent:107:6"]

    success_total = sum(
        1 for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") in SUCCESS_KEYS
        and window_start_id < rec.get("_id", "") <= window_end_id
    )

    neg_total = sum(
        1 for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") in NEG_KEYS
        and window_start_id < rec.get("_id", "") <= window_end_id
    )

    if success_total == 1 and neg_total == 0:
        score += 2
    elif success_total == 1 and neg_total == 1:
        score += 1

    color = "green" if score > 2 else "yellow"

color

'green'

In [53]:
print("latest_start:", latest_start)
print("latest_end:", latest_end)
print("window_start_id:", window_start_id)
print("window_end_id:", window_end_id)
print("c_m1_top:", c_m1_top)
print("c_m1_bottom:", c_m1_bottom)
print("c_m2_floor5:", c_m2_floor5)
print("success_total:", success_total)
print("neg_total:", neg_total)
print("score:", score)
print("color:", color)

latest_start: {'_id': '69b49ef598a42e37064a8328', 'data': {'questEventType': 'questActiveEvent', 'questID': '50', 'questName': 'Power Play - Floor 5'}, 'device': {'dpi': 96, 'gMemory': 512, 'gdApiType': 'OpenGLES3', 'gdName': 'ANGLE (NVIDIA, NVIDIA GeForce RTX 5090 (0x00002B85) Direct3D11 vs_5_0 ps_5_0, D3D11)', 'memory': 209, 'os': 'Windows 10', 'platform': 'UnityWebGL', 'processors': 1, 'resolution': {'height': 1440, 'refreshRate': {'denominator': 1, 'numerator': 60, 'value': 60}, 'width': 3440}}, 'eventKey': 'questActiveEvent:50', 'eventType': 'questEvent', 'game': 'mhs', 'playerId': 'wenyi222', 'sceneName': 'Unit 4 Dev - Dungeon', 'serverTimestamp': '2026-03-13T23:34:13.372Z', 'version': '20260313-10763'}
latest_end: {'_id': '69b4a06c98a42e37064a83d8', 'data': {'questEventType': 'questActiveEvent', 'questID': '36', 'questName': 'Saving Cadet Anderson'}, 'device': {'dpi': 96, 'gMemory': 512, 'gdApiType': 'OpenGLES3', 'gdName': 'ANGLE (NVIDIA, NVIDIA GeForce RTX 5090 (0x00002B85) Dir

## U4 P6
### Without time window

In [40]:
score = 0

# Latest placement for Box 1
box1_candidates = sorted(
    [
        rec for rec in records
        if rec.get("playerId") == pid
        and rec.get("eventType") == "TerasGardenBox"
        and rec.get("data", {}).get("actionType") == "cameraPlaced"
        and rec.get("data", {}).get("boxId") == "0"
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_box1 = box1_candidates[-1] if box1_candidates else None

if latest_box1 and latest_box1.get("data", {}).get("soilType") == "Clay":
    score += 1

# Latest placement for Box 2
box2_candidates = sorted(
    [
        rec for rec in records
        if rec.get("playerId") == pid
        and rec.get("eventType") == "TerasGardenBox"
        and rec.get("data", {}).get("actionType") == "cameraPlaced"
        and rec.get("data", {}).get("boxId") == "1"
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_box2 = box2_candidates[-1] if box2_candidates else None

if latest_box2 and latest_box2.get("data", {}).get("soilType") == "Sand":
    score += 1

# Latest placement for Box 3
box3_candidates = sorted(
    [
        rec for rec in records
        if rec.get("playerId") == pid
        and rec.get("eventType") == "TerasGardenBox"
        and rec.get("data", {}).get("actionType") == "cameraPlaced"
        and rec.get("data", {}).get("boxId") == "2"
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_box3 = box3_candidates[-1] if box3_candidates else None

if latest_box3 and latest_box3.get("data", {}).get("soilType") == "Gravel":
    score += 1

color = "green" if score >= 2 else "yellow"

color

'green'

In [41]:
print("latest_box1:", latest_box1)
print("latest_box2:", latest_box2)
print("latest_box3:", latest_box3)
print("score:", score)
print("color:", color)

latest_box1: {'_id': '69b4a28698a42e37064a84f8', 'data': {'actionType': 'cameraPlaced', 'boxId': '0', 'soilType': 'Clay'}, 'device': {'dpi': 96, 'gMemory': 512, 'gdApiType': 'OpenGLES3', 'gdName': 'ANGLE (NVIDIA, NVIDIA GeForce RTX 5090 (0x00002B85) Direct3D11 vs_5_0 ps_5_0, D3D11)', 'memory': 209, 'os': 'Windows 10', 'platform': 'UnityWebGL', 'processors': 1, 'resolution': {'height': 1440, 'refreshRate': {'denominator': 1, 'numerator': 60, 'value': 60}, 'width': 3440}}, 'eventType': 'TerasGardenBox', 'game': 'mhs', 'playerId': 'wenyi222', 'sceneName': 'Unit 4 Dev', 'serverTimestamp': '2026-03-13T23:49:26.112Z', 'version': '20260313-10763'}
latest_box2: {'_id': '69b4a29998a42e37064a850a', 'data': {'actionType': 'cameraPlaced', 'boxId': '1', 'soilType': 'Sand'}, 'device': {'dpi': 96, 'gMemory': 512, 'gdApiType': 'OpenGLES3', 'gdName': 'ANGLE (NVIDIA, NVIDIA GeForce RTX 5090 (0x00002B85) Direct3D11 vs_5_0 ps_5_0, D3D11)', 'memory': 209, 'os': 'Windows 10', 'platform': 'UnityWebGL', 'proc

### With time window

In [45]:
WINDOW_START_KEY = "questActiveEvent:41"
WINDOW_END_KEY = "questFinishEvent:56"

# 1) Most recent window start
start_candidates = sorted(
    [
        rec for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") == WINDOW_START_KEY
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_start = start_candidates[-1] if start_candidates else None

# 2) Most recent window end
end_candidates = sorted(
    [
        rec for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") == WINDOW_END_KEY
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_end = end_candidates[-1] if end_candidates else None

if latest_start is None or latest_end is None or latest_end["_id"] < latest_start["_id"]:
    color = "yellow"
else:
    window_start_id = latest_start["_id"]
    window_end_id = latest_end["_id"]

    score = 0

    # Latest placement for Box 1 (boxId = "0") within window
    box1_candidates = sorted(
        [
            rec for rec in records
            if rec.get("game") == "mhs"
            and rec.get("playerId") == pid
            and rec.get("eventType") == "TerasGardenBox"
            and rec.get("data", {}).get("actionType") == "cameraPlaced"
            and rec.get("data", {}).get("boxId") == "0"
            and window_start_id < rec.get("_id", "") <= window_end_id
            and "_id" in rec
        ],
        key=lambda x: x["_id"]
    )
    latest_box1 = box1_candidates[-1] if box1_candidates else None

    if latest_box1 and latest_box1.get("data", {}).get("soilType") == "Clay":
        score += 1

    # Latest placement for Box 2 within window
    box2_candidates = sorted(
        [
            rec for rec in records
            if rec.get("game") == "mhs"
            and rec.get("playerId") == pid
            and rec.get("eventType") == "TerasGardenBox"
            and rec.get("data", {}).get("actionType") == "cameraPlaced"
            and rec.get("data", {}).get("boxId") == "1"
            and window_start_id < rec.get("_id", "") <= window_end_id
            and "_id" in rec
        ],
        key=lambda x: x["_id"]
    )
    latest_box2 = box2_candidates[-1] if box2_candidates else None

    if latest_box2 and latest_box2.get("data", {}).get("soilType") == "Sand":
        score += 1

    # Latest placement for Box 3 within window
    box3_candidates = sorted(
        [
            rec for rec in records
            if rec.get("game") == "mhs"
            and rec.get("playerId") == pid
            and rec.get("eventType") == "TerasGardenBox"
            and rec.get("data", {}).get("actionType") == "cameraPlaced"
            and rec.get("data", {}).get("boxId") == "2"
            and window_start_id < rec.get("_id", "") <= window_end_id
            and "_id" in rec
        ],
        key=lambda x: x["_id"]
    )
    latest_box3 = box3_candidates[-1] if box3_candidates else None

    if latest_box3 and latest_box3.get("data", {}).get("soilType") == "Gravel":
        score += 1

    color = "green" if score >= 2 else "yellow"
color

'green'

In [46]:
print("latest_start:", latest_start)
print("latest_end:", latest_end)
print("window_start_id:", window_start_id)
print("window_end_id:", window_end_id)
print("latest_box1:", latest_box1)
print("latest_box2:", latest_box2)
print("latest_box3:", latest_box3)
print("score:", score)
print("color:", color)

latest_start: {'_id': '69b4a24f98a42e37064a84ca', 'data': {'questEventType': 'questActiveEvent', 'questID': '41', 'questName': 'Desert Delicacies'}, 'device': {'dpi': 96, 'gMemory': 512, 'gdApiType': 'OpenGLES3', 'gdName': 'ANGLE (NVIDIA, NVIDIA GeForce RTX 5090 (0x00002B85) Direct3D11 vs_5_0 ps_5_0, D3D11)', 'memory': 209, 'os': 'Windows 10', 'platform': 'UnityWebGL', 'processors': 1, 'resolution': {'height': 1440, 'refreshRate': {'denominator': 1, 'numerator': 60, 'value': 60}, 'width': 3440}}, 'eventKey': 'questActiveEvent:41', 'eventType': 'questEvent', 'game': 'mhs', 'playerId': 'wenyi222', 'sceneName': 'Unit 4 Dev', 'serverTimestamp': '2026-03-13T23:48:31.319Z', 'version': '20260313-10763'}
latest_end: {'_id': '69b4a3ca98a42e37064a85b8', 'data': {'questEventType': 'questFinishEvent', 'questID': '56', 'questName': 'Chief of Reasoning', 'questSuccessOrFailure': 'Succeeded'}, 'device': {'dpi': 96, 'gMemory': 512, 'gdApiType': 'OpenGLES3', 'gdName': 'ANGLE (NVIDIA, NVIDIA GeForce RTX

## U5 P4
### Without time window

In [48]:
SUCCESS_KEY = "DialogueNodeEvent:106:35"

NEGATIVE_KEYS = [
    "DialogueNodeEvent:106:4",
    "DialogueNodeEvent:106:25",
    "DialogueNodeEvent:106:26",
    "DialogueNodeEvent:106:27",
    "DialogueNodeEvent:106:28",
    "DialogueNodeEvent:106:29",
    "DialogueNodeEvent:106:30",
    "DialogueNodeEvent:106:31",
    "DialogueNodeEvent:106:32",
    "DialogueNodeEvent:106:33",
    "DialogueNodeEvent:106:34"
]

has_success = any(
    rec.get("playerId") == pid and rec.get("eventKey") == SUCCESS_KEY
    for rec in records
)

if not has_success:
    color = "yellow"
else:
    cnt = sum(
        1 for rec in records
        if rec.get("playerId") == pid and rec.get("eventKey") in NEGATIVE_KEYS
    )

    color = "green" if cnt == 0 else "yellow"

print("has_success:", has_success)
print("negative_count:", cnt)
print("color:", color)

has_success: True
negative_count: 0
color: green


### With time window

In [50]:
START_KEY = "questFinishEvent:44"
END_KEY = "questFinishEvent:45"
SUCCESS_KEY = "DialogueNodeEvent:106:35"

NEGATIVE_KEYS = [
    "DialogueNodeEvent:106:4",
    "DialogueNodeEvent:106:25",
    "DialogueNodeEvent:106:26",
    "DialogueNodeEvent:106:27",
    "DialogueNodeEvent:106:28",
    "DialogueNodeEvent:106:29",
    "DialogueNodeEvent:106:30",
    "DialogueNodeEvent:106:31",
    "DialogueNodeEvent:106:32",
    "DialogueNodeEvent:106:33",
    "DialogueNodeEvent:106:34"
]

# 1) Latest end anchor
end_candidates = sorted(
    [
        rec for rec in records
        if rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") == END_KEY
        and "_id" in rec
    ],
    key=lambda x: x["_id"]
)

latest_end = end_candidates[-1] if end_candidates else None

if latest_end is None:
    color = "yellow"
else:
    # 2) Previous start anchor before latest end
    start_candidates = sorted(
        [
            rec for rec in records
            if rec.get("game") == "mhs"
            and rec.get("playerId") == pid
            and rec.get("eventKey") == START_KEY
            and "_id" in rec
            and rec["_id"] < latest_end["_id"]
        ],
        key=lambda x: x["_id"]
    )

    prev_start = start_candidates[-1] if start_candidates else None

    window_start_id = prev_start["_id"] if prev_start is not None else "000000000000000000000000"
    window_end_id = latest_end["_id"]

    # 3) Check success node inside window
    has_success = any(
        rec.get("game") == "mhs"
        and rec.get("playerId") == pid
        and rec.get("eventKey") == SUCCESS_KEY
        and window_start_id < rec.get("_id", "") <= window_end_id
        for rec in records
    )

    if not has_success:
        color = "yellow"
    else:
        # 4) Count negative nodes inside window
        cnt = sum(
            1 for rec in records
            if rec.get("game") == "mhs"
            and rec.get("playerId") == pid
            and rec.get("eventKey") in NEGATIVE_KEYS
            and window_start_id < rec.get("_id", "") <= window_end_id
        )

        color = "green" if cnt == 0 else "yellow"

print("latest_end:", latest_end)
print("prev_start:", prev_start)
print("window_start_id:", window_start_id)
print("window_end_id:", window_end_id)
print("has_success:", has_success)
print("negative_count:", cnt)
print("color:", color)

latest_end: {'_id': '69b4a71998a42e37064a8718', 'data': {'questEventType': 'questFinishEvent', 'questID': '45', 'questName': 'Water Problems Require Water Solutions', 'questSuccessOrFailure': 'Succeeded'}, 'device': {'dpi': 96, 'gMemory': 512, 'gdApiType': 'OpenGLES3', 'gdName': 'ANGLE (NVIDIA, NVIDIA GeForce RTX 5090 (0x00002B85) Direct3D11 vs_5_0 ps_5_0, D3D11)', 'memory': 182, 'os': 'Windows 10', 'platform': 'UnityWebGL', 'processors': 1, 'resolution': {'height': 1440, 'refreshRate': {'denominator': 1, 'numerator': 60, 'value': 60}, 'width': 3440}}, 'eventKey': 'questFinishEvent:45', 'eventType': 'questEvent', 'game': 'mhs', 'playerId': 'wenyi222', 'sceneName': 'Unit 5 Dev', 'serverTimestamp': '2026-03-14T00:08:57.652Z', 'version': '20260313-10763'}
prev_start: {'_id': '69b4a63998a42e37064a86b6', 'data': {'questEventType': 'questFinishEvent', 'questID': '44', 'questName': 'WAT Happened Here?', 'questSuccessOrFailure': 'Succeeded'}, 'device': {'dpi': 96, 'gMemory': 512, 'gdApiType': 